# 04 — Numerical-reasoning LLM (FinQA, QLoRA) — stretch goal

| | |
|---|---|
| Base | `Qwen2.5-3B-Instruct` (7B only once 3B is comfortable) |
| Data | `ibm/finqa` |
| Method | QLoRA, 4-bit NF4, LoRA r=16 α=32, attention + MLP projections |
| T4 | batch 1, grad-accum 8–16, max_len 1024, gradient checkpointing, paged AdamW |

FinQA supplies multi-step reasoning programs over financial tables — the right
supervision for the derived-KPI and reasoning steps.

**The result worth reporting is the three-way comparison** on identical prompts:
base zero-shot, this QLoRA model, and a hosted frontier model. "Reached X% of the
hosted model's execution accuracy at zero marginal cost" is a strong, honest
claim; "our model scored X%" on its own is not.

QLoRA is the cheap checkpointing case: only adapter weights are saved (~50–100MB),
so this can checkpoint frequently at negligible cost.

In [ ]:
# Confirm we actually have the T4 this recipe is written for.
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

import torch
assert torch.cuda.is_available(), "No GPU. Runtime > Change runtime type > T4 GPU."
print(f"torch {torch.__version__} | {torch.cuda.get_device_name(0)} | "
      f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Checkpoints MUST live somewhere that survives the VM (section 5.5).
# /content is ephemeral - it vanishes with the runtime, which is exactly the
# failure checkpointing exists to defend against.
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/affa'
os.makedirs(DRIVE_ROOT, exist_ok=True)
print('checkpoints ->', DRIVE_ROOT)

In [ ]:
# Clone or update the repo, and verify it is current. Re-running this notebook
# from the top after a disconnect must not silently train an old revision.
import os, subprocess

REPO_URL = 'https://github.com/abhinaba01/agentic-financial-filing-analysis.git'
REPO_DIR = '/content/agentic-financial-filing-analysis'

if not os.path.isdir(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', '--all'], check=True)

local  = subprocess.run(['git', '-C', REPO_DIR, 'rev-parse', 'HEAD'],
                        capture_output=True, text=True).stdout.strip()
remote = subprocess.run(['git', '-C', REPO_DIR, 'rev-parse', '@{u}'],
                        capture_output=True, text=True).stdout.strip()

if remote and local != remote:
    print(f'repo is BEHIND origin (local {local[:8]} != remote {remote[:8]})')
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'], check=True)
    print('pulled; RESTART THE RUNTIME so the new code is imported')
else:
    print(f'repo is current at {local[:8]}')

os.chdir(REPO_DIR)

In [ ]:
%pip install -q -e ".[train,eval,hosted]"
%pip install -q "datasets>=2.19,<4.0" bitsandbytes peft accelerate

import datasets, transformers, peft
print('datasets', datasets.__version__, '| transformers', transformers.__version__,
      '| peft', peft.__version__)
assert int(datasets.__version__.split('.')[0]) < 4

In [ ]:
SEED          = 42
BASE_MODEL    = 'Qwen/Qwen2.5-3B-Instruct'
TRAIN_SAMPLES = None
EVAL_SAMPLES  = 200
MAX_LENGTH    = 1024
BATCH_SIZE    = 1
GRAD_ACCUM    = 16     # effective batch 16
EPOCHS        = 2
LR            = 2e-4   # higher than full fine-tuning: standard for LoRA
SAVE_STEPS    = 100    # adapters are tiny, so save often

CKPT_DIR = f'{DRIVE_ROOT}/finqa_qlora'
print(CKPT_DIR)

## Checkpointing and resume

Colab runtimes disconnect, get recycled, and hit idle timeouts. Everything below
is built so a crash costs minutes, not the whole run.

**What resume restores:** model weights, optimizer moments, LR-scheduler
position, RNG state, global step, and dataloader position. That is why we resume
rather than "just train again from the saved weights" — restarting the optimizer
and the LR schedule from scratch is a *different run*, and its loss curve will
not join up with the first half.

**The cell below is idempotent.** Re-run it after a crash and it resumes
automatically, with no code edit.

**Determinism is a precondition.** `SEED`, `TRAIN_SAMPLES` and `EVAL_SAMPLES` are
written into the checkpoint directory as JSON, and the resume path *refuses* to
continue if they no longer match. Changing any of them after a crash means the
global step now points into different data and the resumed run is silently
meaningless (anti-pattern #14).

**Disk:** a full checkpoint is roughly 3–4× model size — fp32 weights plus two
AdamW moments — so `save_total_limit=2` is required, not tidiness, against
Drive's 15GB free tier. `save_steps` is set for ~15–20 minutes of training, not
per epoch: an epoch here is 40+ minutes and a disconnect at minute 39 loses all
of it.

In [ ]:
!python training/train_finqa_qlora.py \
    --output-dir "{CKPT_DIR}" \
    --base-model {BASE_MODEL} \
    --seed {SEED} \
    --eval-samples {EVAL_SAMPLES} \
    --max-length {MAX_LENGTH} \
    --batch-size {BATCH_SIZE} \
    --grad-accum {GRAD_ACCUM} \
    --epochs {EPOCHS} \
    --learning-rate {LR} \
    --save-steps {SAVE_STEPS}

## Test the resume path — do not assume it

Untested resume logic is usually broken resume logic, and the moment you find
out is the moment you have already lost the run.

1. Run the training cell above and let it write at least two checkpoints.
2. **Runtime → Interrupt execution** (or just let the runtime die).
3. Re-run the training cell *unchanged*.

What you should see: `resuming from .../checkpoint-N`, and the loss continuing
from where it stopped rather than restarting near its initial value. If step
numbering restarts at 0, resume is not working — fix that before starting the
real run.

## The three-way comparison

The architecture puts the local fine-tune and the hosted API model behind one
interface, selectable by config — so this is a flag, not a rewrite.

`--hosted` costs API calls. Set `ANTHROPIC_API_KEY` (or `OPENAI_API_KEY`) first,
and keep `--limit` small while you are iterating.

In [ ]:
# Point the config at the adapter you just trained.
import yaml, pathlib

cfg_path = pathlib.Path('configs/default.yaml')
cfg = yaml.safe_load(cfg_path.read_text())
cfg['models']['reasoner']['backend'] = 'local'
cfg['models']['reasoner']['local_adapter'] = f'{CKPT_DIR}/final'
cfg_path.write_text(yaml.safe_dump(cfg, sort_keys=False))
print(yaml.safe_dump(cfg['models']['reasoner'], sort_keys=False))

In [ ]:
import os
os.environ['ANTHROPIC_API_KEY'] = ''  # paste yours, or drop --hosted below

# Measures: this adapter, the base model zero-shot, and the hosted model,
# on identical prompts and the same split.
!affa-eval finqa --hosted --limit 200 --output eval_results/finqa.json

import json
r = json.load(open('eval_results/finqa.json'))
print('adapter ', r['metrics'])
print('base    ', r['baseline_metrics'])
for note in r['notes']:
    print(' -', note)

In [ ]:
# Push the model and a card carrying the REAL numbers and the subset size.
# A model card with aspirational numbers is worse than no card.
from huggingface_hub import notebook_login
notebook_login()

HUB_ID = 'YOUR_USERNAME/affa-finqa-qlora'

card = """---
license: apache-2.0
tags: [finance, sec-filings, affa]
---

# affa-finqa-qlora

Fine-tuned for the Agentic Financial Filing Analyst.

QLoRA adapter for `Qwen2.5-3B-Instruct`, trained on FinQA multi-step numerical reasoning over financial tables.

## Measured results

Fill these in from the evaluation cell above. Report the **test** split score,
the **baseline measured on the same data with the same protocol**, and the
training subset size. Do not paste a number from a paper here.

| metric | this model | baseline | notes |
|---|---:|---:|---|
| (fill in) | | | |

- Training subset: `TRAIN_SAMPLES` (state the number actually used)
- Seed: `SEED`
- Checkpoint selected on: validation split
- Test split touched: once

## Not financial advice

Research and educational use only.
"""

import pathlib
pathlib.Path(f'{CKPT_DIR}/final/README.md').write_text(card, encoding='utf-8')
print('model card written; review it before pushing')